<a href="https://colab.research.google.com/github/AllyJT/eda202602/blob/main/EDA202602.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory Data Analysis (EDA) with Income and Australian Cities  

This notebook walks through a typical EDA workflow using a small **synthetic** dataset (for teaching) containing:  
- **Primary numerical feature:** `Income_AUD` (annual income in **AUD**)  
- **Primary categorical feature:** `City` (Australian cities)  

Along the way we cover:  
1. Data loading and basic checks  
2. Data cleaning and transformation (normalisation/standardisation)  
3. Encoding categorical features (one-hot and ordinal encoding)  
4. Feature selection ideas  
5. Outlier analysis (IQR, Z-score, robust methods)  
6. Visualisation (histograms, line charts, pair plots, correlation plots)  

> **Note:** The dataset intentionally includes missing values, outliers, and a few invalid entries to mimic real-world data.


In [ ]:
# If running in a fresh environment, you may need these:
#!pip install pandas numpy matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.feature_selection import mutual_info_regression

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


## 0) Load data
Don't worry about this lengthy code block, just run it and load our data!

In [ ]:
rng = np.random.default_rng(42)
n_rows = 1200

cities = np.array([
    "Sydney", "Melbourne", "Brisbane", "Perth",
    "Adelaide", "Canberra", "Hobart", "Darwin"
])
education_levels = np.array(["High School", "TAFE/Cert", "Bachelor", "Postgrad"])
occupations = np.array(["Administration", "Trades", "Education", "Healthcare", "Technology", "Retail"])

city_index = rng.integers(0, len(cities), n_rows)
education_index = rng.choice(4, n_rows, p=[0.25, 0.25, 0.35, 0.15])
occupation_index = rng.integers(0, len(occupations), n_rows)
age = rng.integers(18, 71, n_rows)
year = rng.integers(2018, 2026, n_rows)

city_income_effect = np.array([12000, 8000, 3000, 6000, 0, 14000, -3000, 4000])
occupation_effect = np.array([0, 5000, 4000, 10000, 18000, -8000])
income = (
    30000 + 550 * (age - 18) + 11000 * education_index
    + city_income_effect[city_index] + occupation_effect[occupation_index]
    + 1800 * (year - 2018) + rng.normal(0, 13000, n_rows)
)
income = np.maximum(income, 15000)
weekly_rent = np.array([650, 550, 480, 490, 420, 570, 410, 470])[city_index]
weekly_rent = np.maximum(150, weekly_rent + 12 * (year - 2018) + rng.normal(0, 80, n_rows))
savings = np.clip(0.08 + (income - 60000) / 400000 - weekly_rent / 10000
                  + rng.normal(0, 0.05, n_rows), -0.10, 0.55)

df = pd.DataFrame({
    "Income_AUD": np.round(income, 2),
    "City": cities[city_index],
    "Rent_AUD": np.round(weekly_rent, 2),
    "Age": age,
    "HouseholdSize": rng.choice([1, 2, 3, 4, 5, 6], n_rows,
                                 p=[0.20, 0.32, 0.20, 0.18, 0.07, 0.03]).astype(float),
    "CommuteMin": np.round(np.clip(rng.normal(32, 15, n_rows), 2, 110), 1),
    "SavingsRate": np.round(savings, 3),
    "Year": year,
    "Education": education_levels[education_index],
    "Occupation": occupations[occupation_index],
})

income_rows = rng.permutation(n_rows)
df.loc[income_rows[:12], "Income_AUD"] = np.round(rng.uniform(300000, 650000, 12), 2)
df.loc[income_rows[12:48], "Income_AUD"] = np.nan

for column in ["Rent_AUD", "CommuteMin", "HouseholdSize", "SavingsRate"]:
    df.loc[rng.choice(n_rows, 24, replace=False), column] = np.nan
for column in ["City", "Education", "Occupation"]:
    df.loc[rng.choice(n_rows, 18, replace=False), column] = np.nan

df.loc[rng.choice(n_rows, 10, replace=False), "CommuteMin"] = -5.0
df.loc[rng.choice(n_rows, 8, replace=False), "HouseholdSize"] = 0.0

df.head()


## 1) Basic understanding of the dataset

A typical first step in EDA is to:
- Inspect a few rows
- Look at data types
- Summarise missingness
- Compute quick descriptive stats

These steps help you catch issues early (e.g., strings that should be numbers, unexpected missing values).


In [ ]:
df.info()


In [ ]:
# Missing values per column
missing = df.isna().sum().sort_values(ascending=False)
missing

In [ ]:
# Descriptive statistics (numerical)
df.describe(include=[np.number]).T

In [ ]:
# Categorical summaries
for col in ["City", "Education", "Occupation"]:
    print(f"\n--- {col} ---")
    display(df[col].value_counts().head(10))

## 2) Data cleaning (fix invalid values)

In real datasets, you often find values that are **impossible** or **out of range**.

Examples in this dataset:
- `CommuteMin` has some negative values (invalid)
- `HouseholdSize` has some zeros (invalid)

A simple strategy is to convert invalid values to `NaN` so they can be handled consistently by imputation rules.


In [ ]:
df_clean = df.copy()

# Rule-based cleaning
invalid_commute = df_clean["CommuteMin"] < 0
invalid_hh = df_clean["HouseholdSize"] < 1

df_clean.loc[invalid_commute, "CommuteMin"] = np.nan
df_clean.loc[invalid_hh, "HouseholdSize"] = np.nan

print("Invalid commute values fixed:", invalid_commute.sum())
print("Invalid household sizes fixed:", invalid_hh.sum())

(df_clean[["CommuteMin", "HouseholdSize"]].isna().sum())

## 3) Data transformation (normalisation & standardisation)

Two very common transformations for numerical features:

### **Standardisation (Z-score scaling)**
Transforms data to have **mean = 0** and **standard deviation = 1**.

### **Normalisation (Min-Max scaling)**
Transforms data to lie within a fixed range (commonly **0 to 1**).

We'll demonstrate both on the income feature.


In [ ]:
income = df_clean[["Income_AUD"]].copy()

# Impute missing income just for demonstrating scaling
income_imputed = income.fillna(income.median()) # finds all the missing data in the "Income" column and replaces it with the median income so that there are no missing values

scaler_z = StandardScaler() # setting up Standard Scaler (Standardisation)
scaler_mm = MinMaxScaler() # Setting up Min-Max (Normalisation)

income_z = scaler_z.fit_transform(income_imputed) # fits the Standard Scaler to income
income_mm = scaler_mm.fit_transform(income_imputed) # fits the Min-Max Scaler to income

# Making a new dataset of only the incomes that were transformed
scaled = pd.DataFrame({
    "Income_AUD": income_imputed["Income_AUD"].values,
    "Income_Z": income_z.flatten(),
    "Income_MinMax": income_mm.flatten()
})

# having a look at the first 5 rows
scaled.head()

Plotting Data using histograms with the Standard Scaled and the Normalised Income Data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(scaled["Income_AUD"], kde=True, ax=axes[0])
axes[0].set_title("Income (raw)")

sns.histplot(scaled["Income_Z"], kde=True, ax=axes[1])
axes[1].set_title("Income (standardised Z)")

sns.histplot(scaled["Income_MinMax"], kde=True, ax=axes[2])
axes[2].set_title("Income (min-max 0–1)")

# define comparable x-limits using the 1st and 99th percentiles of the raw income
p_low, p_high = np.nanpercentile(scaled["Income_AUD"], [1, 99])

# map to Z (standardised) scale
mean_raw = np.nanmean(scaled["Income_AUD"])
std_raw = np.nanstd(scaled["Income_AUD"])
if std_raw == 0:
    z_low, z_high = -3, 3
else:
    z_low = (p_low - mean_raw) / std_raw
    z_high = (p_high - mean_raw) / std_raw

# map to MinMax (0-1) scale
min_raw = np.nanmin(scaled["Income_AUD"])
max_raw = np.nanmax(scaled["Income_AUD"])
if max_raw == min_raw:
    mm_low, mm_high = 0.0, 1.0
else:
    mm_low = (p_low - min_raw) / (max_raw - min_raw)
    mm_high = (p_high - min_raw) / (max_raw - min_raw)

axes[0].set_xlim(p_low, p_high)
axes[1].set_xlim(z_low, z_high)
axes[2].set_xlim(mm_low, mm_high)

plt.tight_layout()
plt.show()

## 4) Encoding categorical features

Most models expect numbers, so we convert categories into numeric representations, or else you wouldn't know what to do with the data.

- **One-hot encoding** for categories with no natural order (`City`, `Occupation`). It converts each unique category into a value (1) if its true and 0 otherwise.
- **Ordinal encoding** for ordered categories (`Education`), so only if there is a clear and equal hierarchy, e.g. Low, Medium, High

We'll one-hot encode `City` and ordinal encode `Education`.


In [ ]:
education_order = [["High School", "TAFE/Cert", "Bachelor", "Postgrad"]]

numeric_cols = ["Income_AUD", "Rent_AUD", "Age", "HouseholdSize", "CommuteMin", "SavingsRate", "Year"]
nominal_cols = ["City", "Occupation"]
ordinal_cols = ["Education"]

# Encoding Steps
# Numerical Data: Impute missing values with the median and then standardise (Z-score scaling)
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Nominal Data: Impute missing values with the most frequent category and then one-hot encode
nominal_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")) # One-Hot Encoder Step here
])

# Ordinal Data: Impute missing values with the most frequent category and then ordinal encode
ordinal_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=education_order))
])

# Combine all transformations into a single preprocessing step to do it all at once.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("nom", nominal_pipe, nominal_cols),
        ("ord", ordinal_pipe, ordinal_cols)
    ],
    remainder="drop"
)

# Apply Changes to the Dataset
X = df_clean[numeric_cols + nominal_cols + ordinal_cols]
X_transformed = preprocessor.fit_transform(X)
X_transformed.shape # having a quick look to see if any data has been lost (luckily, no)

In [ ]:
num_names = numeric_cols
nom_names = list(preprocessor.named_transformers_["nom"].named_steps["onehot"].get_feature_names_out(nominal_cols))
ord_names = ordinal_cols

all_feature_names = num_names + nom_names + ord_names

print("Total features after encoding:", len(all_feature_names))
print(all_feature_names[:15])

## 5) Feature selection (which features matter?)

Here we treat **income** as a target to demonstrate selection.

We'll show:
1. **Correlation** (numerical ↔ numerical, linear)
2. **Mutual information** (more general, works with mixed features) (2nd Year Data Science Subject Matter)


In [ ]:
num_df = df_clean[numeric_cols].copy()
num_df = num_df.fillna(num_df.median(numeric_only=True))

corr = num_df.corr(numeric_only=True)

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation (numerical features)")
plt.show()

In [ ]:
y = df_clean["Income_AUD"].copy().fillna(df_clean["Income_AUD"].median()) # cleaning data once again

mi = mutual_info_regression(X_transformed, y, random_state=42) # Mutual Information Calculation on all categories
mi_s = pd.Series(mi, index=all_feature_names).sort_values(ascending=False)

# to interpret the mutual information scores, we can look at the top features that have the highest scores, which indicate a stronger relationship with the target variable (Income_AUD).
mi_s.head(15)

In [ ]:
plt.figure(figsize=(10, 6))
mi_s.head(20).iloc[::-1].plot(kind="barh")
plt.title("Top 20 features by mutual information with Income")
plt.xlabel("Mutual Information")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 6) Outlier analysis and handling

We'll explore **income** outliers using:
- **IQR rule** (Tukey fences)
- **Z-score rule**
- **Robust Z-score** (MAD)

Then we create a *trimmed* dataset after removing IQR outliers.
Why do we care about outliers? Messes up calculation of things that are affected by large/small numbers like Averages.


In [ ]:
income_series = df_clean["Income_AUD"].copy()
income_non_missing = income_series.dropna()

Q1 = income_non_missing.quantile(0.25)
Q3 = income_non_missing.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_iqr = income_non_missing[(income_non_missing < lower) | (income_non_missing > upper)]

lower, upper, outliers_iqr.shape[0]

In [ ]:
z = (income_non_missing - income_non_missing.mean()) / income_non_missing.std()
outliers_z = income_non_missing[np.abs(z) > 3]

outliers_z.shape[0]

In [ ]:
median = income_non_missing.median()
mad = np.median(np.abs(income_non_missing - median))
robust_z = 0.6745 * (income_non_missing - median) / mad
outliers_mad = income_non_missing[np.abs(robust_z) > 3.5]

outliers_mad.shape[0]

Industry Standard for Identifying Outliers (manually) is typicall if the data is 1.5x more outside IQR (Tukey's Rule)

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x=df_clean["Income_AUD"], color="#7aa6c2")
plt.title("Income boxplot (shows outliers)")
plt.xlabel("Income_AUD")
plt.show() # box plot with outliers

plt.figure(figsize=(10, 4))
sns.histplot(df_clean["Income_AUD"], kde=True, bins=40)
plt.title("Income histogram (raw)")
plt.xlabel("Income_AUD")
plt.show()

In [ ]:
trim_mask = (df_clean["Income_AUD"].isna()) | ((df_clean["Income_AUD"] >= lower) & (df_clean["Income_AUD"] <= upper))
df_trim = df_clean.loc[trim_mask].copy()

print("Original rows:", len(df_clean))
print("Trimmed rows:", len(df_trim))

summary_compare = pd.DataFrame({
    "Original": df_clean["Income_AUD"].describe(),
    "Trimmed": df_trim["Income_AUD"].describe()
})
summary_compare

## 7) Plotting & visualising data

We'll create:
- Histograms (distribution)
- Boxplots by city (group comparison + outliers)
- A line chart (trend by year)
- Pair plot (multivariate relationships)
- Correlation heatmap


In [ ]:
g = sns.FacetGrid(df_clean, col="City", col_wrap=4, height=3, sharex=False, sharey=False)
g.map_dataframe(sns.histplot, x="Income_AUD", bins=25)

g.fig.suptitle("Income distribution by city", y=1.02)
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
order = df_clean.groupby("City")["Income_AUD"].median().sort_values().index
sns.boxplot(data=df_clean, x="City", y="Income_AUD", order=order)
plt.title("Income by city (boxplot)")
plt.xticks(rotation=35, ha="right")
plt.show()

In [ ]:
trend = df_clean.groupby(["Year", "City"], as_index=False)["Income_AUD"].median()

plt.figure(figsize=(12, 6))
sns.lineplot(data=trend, x="Year", y="Income_AUD", hue="City", marker="o")
plt.title("Median income over time by city")
plt.ylabel("Median Income (AUD)")
plt.show()

In [ ]:
sample = df_clean.sample(600, random_state=42).copy()
sample_num = sample[["Income_AUD", "Rent_AUD", "Age", "CommuteMin", "SavingsRate"]].copy()
sample_num = sample_num.fillna(sample_num.median())

sample_plot = sample[["City"]].join(sample_num)

sns.pairplot(sample_plot, hue="City", corner=True, plot_kws={"alpha": 0.4, "s": 20})
plt.suptitle("Pair plot of key numerical features (sample)", y=1.02)
plt.show()

In [ ]:

num_trim = df_clean[numeric_cols].fillna(df_clean[numeric_cols].median(numeric_only=True))

plt.figure(figsize=(9, 7))
sns.heatmap(num_trim.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation (trimmed dataset)")
plt.show()

## 8) Wrap-up: Typical EDA workflow recap

1. **Load & inspect** (data types, missingness, quick stats)
2. **Clean** (invalid values → `NaN`, decide missing-value strategy)
3. **Transform** (standardise/normalise as needed)
4. **Encode categories** (one-hot / ordinal)
5. **Explore** (plots + group summaries)
6. **Outliers** (understand vs remove vs cap)
7. **Feature selection** (correlation + mutual information)

### Student exercises
- Compare results using raw vs trimmed dataset
- Try different outlier thresholds
- Swap `StandardScaler` for `MinMaxScaler` in the preprocessing pipeline
- Add a simple predictive model (e.g., linear regression) and evaluate changes
